# Validation: the integrators against a Kepler orbit

Every figure in this notebook is a comparison against a problem with a known
answer, and every one of them is asserted before it is plotted. If a cell runs,
its claim held.

The instrument throughout is a two-body orbit. It is constructed rather than
sampled, released at periapsis, and its exact solution after one period is the
state it started in. That makes the difference between the integrated state and
the initial state the whole of the method's error over an orbit and nothing
else.

The same three results are in the C++ test suite, in
`tests/integrators/`. Reproducing them here is the point: it is what says the
bindings expose the library rather than a version of it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import orrery

print(f"orrery {orrery.__version__}, scalar type {orrery.dtype}")

## The orbit

`KeplerParameters` describes it and `kepler_period` gives the period Kepler's
third law assigns it, `2 pi sqrt(a^3 / G M)`, in units where `G` is one.

In [ ]:
orbit = orrery.KeplerParameters(
    primary_mass=1.0, secondary_mass=1.0, semi_major_axis=1.0, eccentricity=0.5
)

period = orrery.kepler_period(orbit)
print(f"period {period:.12f}")
print(f"energy {orrery.kepler_energy(orbit):.12f}")
print(f"periapsis separation {orrery.kepler_periapsis_distance(orbit):.12f}")

# The constructed state has to have the energy and angular momentum the closed
# form says, or nothing after this means anything.
start = orrery.kepler_orbit(orbit)
measured = orrery.measure_diagnostics(start, 0.0)
assert np.isclose(measured.total_energy, orrery.kepler_energy(orbit))
assert np.isclose(orrery.norm(measured.angular_momentum), orrery.kepler_angular_momentum(orbit))
print("the constructed state matches the closed form")

## A run, and the path it traces

A configuration is the same record the configuration file parses into, so this
cell describes exactly what `orrery run` would.

In [ ]:
def kepler_configuration(integrator, steps_per_orbit, orbits, eccentricity=0.5):
    configuration = orrery.Configuration()
    configuration.initial_conditions.kind = orrery.InitialConditionKind.kepler
    configuration.initial_conditions.primary_mass = orbit.primary_mass
    configuration.initial_conditions.secondary_mass = orbit.secondary_mass
    configuration.initial_conditions.semi_major_axis = orbit.semi_major_axis
    configuration.initial_conditions.eccentricity = eccentricity
    configuration.solver.kind = orrery.SolverKind.direct
    configuration.solver.softening = 0.0
    configuration.integrator.kind = integrator
    configuration.run.timestep = period / steps_per_orbit
    configuration.run.steps = steps_per_orbit * orbits
    return configuration


configuration = kepler_configuration(orrery.IntegratorKind.yoshida4, 512, 3)
simulation = orrery.assemble(configuration)

# The state is read without copying. `simulation.particles.position_x` is a
# NumPy view of the array the solver is writing into, so the loop below costs
# two floats a step rather than a copy of the state.
x = simulation.particles.position_x
y = simulation.particles.position_y

path = np.empty((configuration.run.steps, 2))
for step in range(configuration.run.steps):
    simulation.step()
    path[step] = (x[1] - x[0], y[1] - y[0])

figure, axes = plt.subplots(figsize=(5, 5))
axes.plot(path[:, 0], path[:, 1], linewidth=0.8)
axes.plot([0], [0], marker="+", color="black")
axes.set_aspect("equal")
axes.set_title(f"separation vector, {configuration.run.steps} steps, e = 0.5")
axes.set_xlabel("x")
axes.set_ylabel("y")
plt.show()

## Result 1: each method converges at the order it claims

Halving the timestep divides the error of a method of order `p` by `2^p`, so the
base-two logarithm of the ratio of two errors is the measured order. The error
is measured as the distance between the state after one period and the state it
started in, which for a circular orbit is exactly the integrator's error.

In [ ]:
def closure_error(integrator, steps_per_orbit):
    """How far one circular orbit fails to return to where it started."""
    configuration = kepler_configuration(integrator, steps_per_orbit, 1, eccentricity=0.0)
    simulation = orrery.assemble(configuration)
    before = orrery.stacked(simulation.particles)
    simulation.run(configuration.run.steps)
    return float(np.max(np.abs(orrery.stacked(simulation.particles) - before)))


methods = [
    ("velocity Verlet", orrery.IntegratorKind.velocity_verlet, 2, 256),
    ("Yoshida 4", orrery.IntegratorKind.yoshida4, 4, 128),
    ("RK4", orrery.IntegratorKind.rk4, 4, 128),
]

print(f"{'method':16} {'stated':>7} {'measured':>9}")
for name, kind, order, coarse in methods:
    coarse_error = closure_error(kind, coarse)
    fine_error = closure_error(kind, 2 * coarse)
    measured_order = np.log2(coarse_error / fine_error)
    print(f"{name:16} {order:7} {measured_order:9.4f}")
    assert abs(measured_order - order) < 0.35, name

print()
print("each method converges at the order it claims")

## Result 2: bounded against secular energy error

This is the comparison the project exists to produce, and it is the reason RK4
is in the library at all.

Velocity Verlet is second order and costs one force evaluation a step. RK4 is
fourth order and costs four. Over four hundred orbits the second-order
symplectic method returns the energy error it began with, oscillating inside a
fixed envelope; the fourth-order non-symplectic one starts far more accurate and
grows without bound.

In [ ]:
def energy_error_history(integrator, orbits=400, steps_per_orbit=200, chunk=197):
    """The relative energy error, sampled through a long run.

    The sampling interval is deliberately not a whole number of orbits. A
    symplectic method's energy error oscillates over each orbit inside a fixed
    envelope, so measuring it once per orbit samples one orbital phase for ever
    and reports the envelope's value at that phase rather than the envelope. It
    is an aliasing mistake and it is an easy one to make: the aliased series
    looks like a slow drift, which is exactly the thing this comparison is
    supposed to distinguish. 197 steps against 200 to the orbit walks the phase
    around instead.
    """
    configuration = kepler_configuration(integrator, steps_per_orbit, orbits)
    simulation = orrery.assemble(configuration)

    reference = simulation.measure().total_energy

    times, errors = [], []
    while simulation.step_index < configuration.run.steps:
        simulation.run(min(chunk, configuration.run.steps - simulation.step_index))
        times.append(simulation.time / period)
        errors.append(abs((simulation.measure().total_energy - reference) / reference))
    return np.array(times), np.array(errors)


histories = {
    name: energy_error_history(kind)
    for name, kind, _, _ in methods
}

figure, axes = plt.subplots(figsize=(8, 4.5))
for name, (times, errors) in histories.items():
    axes.semilogy(times, errors, label=name)
axes.set_xlabel("orbits")
axes.set_ylabel("relative energy error")
axes.set_title("bounded against secular: the whole of ADR-0011 in one plot")
axes.legend()
axes.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
print(f"{'method':16} {'first 5%':>12} {'last 5%':>12} {'growth':>8}")
for name, (times, errors) in histories.items():
    edge = max(1, len(errors) // 20)
    first, last = errors[:edge].mean(), errors[-edge:].mean()
    print(f"{name:16} {first:12.4e} {last:12.4e} {last / first:8.2f}x")

# The symplectic pair end where they began. RK4 does not, and is still climbing
# when the run stops.
for name in ("velocity Verlet", "Yoshida 4"):
    errors = histories[name][1]
    edge = max(1, len(errors) // 20)
    assert errors[-edge:].mean() < 2.0 * errors[:edge].mean(), name
    assert errors.max() < 2.0 * errors[:edge].mean(), name

rk4_errors = histories["RK4"][1]
edge = max(1, len(rk4_errors) // 20)
assert rk4_errors[-edge:].mean() > 5.0 * rk4_errors[:edge].mean()
assert np.all(np.diff(rk4_errors[-edge:]) > 0.0)

print()
print("the symplectic methods are bounded; RK4 grows and is still growing")

## Result 3: a sampled Plummer sphere is in virial equilibrium

The virial theorem says a self-gravitating system in equilibrium has
`2T / |U| = 1`. The sampler draws positions and speeds from the model's
distribution functions and never computes that ratio, so measuring it afterwards
is a check on the sampler rather than a restatement of it.

The closed-form energies of the model are the second check, and the third is
that a Plummer sphere at the default scale radius is in standard N-body units,
where the total energy is exactly `-1/4`.

In [ ]:
seed = 20260812
counts = [256, 1024, 4096, 16384]

print(f"{'N':>7} {'virial ratio':>14} {'E measured':>14} {'E closed form':>14}")
ratios = []
for count in counts:
    parameters = orrery.PlummerParameters(count=count)
    sample = orrery.plummer_sphere(parameters, seed=seed)
    diagnostics = orrery.measure_diagnostics(sample, 0.0)
    ratios.append(diagnostics.virial_ratio)
    print(
        f"{count:7} {diagnostics.virial_ratio:14.4f} {diagnostics.total_energy:14.6f} "
        f"{orrery.plummer_total_energy(parameters):14.6f}"
    )

assert np.isclose(orrery.plummer_total_energy(orrery.PlummerParameters(count=1)), -0.25)
assert abs(ratios[-1] - 1.0) < 0.05, f"seed {seed}"
print()
print("the sampled sphere is in virial equilibrium, and in standard N-body units")

In [ ]:
parameters = orrery.PlummerParameters(count=20000)
sample = orrery.plummer_sphere(parameters, seed=seed)

radii = orrery.radii(sample)
a = parameters.scale_radius

# The cumulative mass profile of a Plummer model, against the sample drawn from
# it. Nothing fitted: the curve is the model and the steps are the draw.
figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))

edges = np.logspace(-1.5, 1.0, 60)
enclosed = np.searchsorted(np.sort(radii), edges) / len(radii)
model = edges**3 / (edges**2 + a**2) ** 1.5

axes[0].loglog(edges, enclosed, label="sample", linewidth=2)
axes[0].loglog(edges, model, "--", label="Plummer model", linewidth=1.5)
axes[0].set_xlabel("radius")
axes[0].set_ylabel("enclosed mass fraction")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

x, y, _ = orrery.components(sample)
axes[1].hexbin(x, y, gridsize=70, extent=(-3, 3, -3, 3), bins="log", cmap="magma")
axes[1].set_aspect("equal")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title(f"{len(sample)} particles, seed {seed}")

plt.tight_layout()
plt.show()

# The sample follows the model it was drawn from, over the range where the bins
# hold enough particles to say so.
inside = (enclosed > 0.02) & (enclosed < 0.98)
assert np.max(np.abs(enclosed[inside] - model[inside])) < 0.02, f"seed {seed}"
print("the sample follows the model's cumulative mass profile")

## What was shown

- Each of the three integrators converges at the order it claims, measured
  rather than asserted.
- The two symplectic methods hold their energy error inside a bounded envelope
  over four hundred orbits; RK4, of the same order as Yoshida and costing a
  third more per step, grows without limit and is still growing at the end.
- A sampled Plummer sphere is in virial equilibrium and has the energy the model
  states in closed form.

Every assertion in this notebook also exists in the C++ test suite. Run it with
`ctest --preset release`.